In [1]:
pip install tensorflow

In [2]:
pip install keras

In [7]:
import os
import numpy as np
from keras.models import Sequential
from keras.layers import Activation, Dropout, Flatten, Dense
from keras.preprocessing.image import ImageDataGenerator
from keras.layers import Convolution2D, MaxPooling2D, ZeroPadding2D
from keras import optimizers
from keras.callbacks import TensorBoard, ModelCheckpoint,EarlyStopping

# dimensions of our images.
img_width, img_height = 150, 150

train_data_dir = "/content/drive/MyDrive/Colab_Notebooks/png/cross_toe_touch-Copy/train_data"
validation_data_dir = "/content/drive/MyDrive/Colab_Notebooks/png/cross_toe_touch-Copy/test_data"

# used to rescale the pixel values from [0, 255] to [0, 1] interval
datagen = ImageDataGenerator(rescale=1./255)

# automagically retrieve images and their classes for train and validation sets
train_generator = datagen.flow_from_directory(
        train_data_dir,
        target_size=(img_width, img_height),
        batch_size=16,
        class_mode='categorical',
        shuffle=True)

validation_generator = datagen.flow_from_directory(
        validation_data_dir,
        target_size=(img_width, img_height),
        batch_size=32,
        class_mode='categorical',
        shuffle=True)

model = Sequential()
model.add(Convolution2D(32, 3, 3, input_shape=(img_width, img_height,3)))
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Convolution2D(32, 3, 3))
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Convolution2D(32, 3, 3))
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Convolution2D(32, 3, 3))
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Flatten())

model.add(Dense(64))
model.add(Activation('relu'))
model.add(Dropout(0.5))

model.add(Dense(3))
model.add(Activation('softmax'))#sigmoid

tbCallBack=[]
tbCallBack=TensorBoard(log_dir='./Graph_Adam32')
tbCallBackchptk=ModelCheckpoint('models/checkpoints/weightsAdam32.h5',save_weights_only=True)
tbCallBackearlyStop=EarlyStopping(patience=3)

model.compile(loss='categorical_crossentropy',
              optimizer='adam', #rmsprop
              metrics=['accuracy'])
#model.summary()
nb_epoch = 20
nb_train_samples = 37500#9576#2048
nb_validation_samples = 1245#1245#832

model.fit_generator(
        train_generator,
        samples_per_epoch=nb_train_samples,
        nb_epoch=nb_epoch,
        validation_data=validation_generator,
        nb_val_samples=nb_validation_samples, callbacks=[tbCallBack,tbCallBackchptk,tbCallBackearlyStop])
model.save('models/model3classesDogCatBirdAdam32.h5')
print(model.evaluate_generator(validation_generator, nb_validation_samples))

Found 8 images belonging to 2 classes.
Found 4 images belonging to 2 classes.


ValueError: ignored

In [3]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import img_to_array, load_img

# Define the directories where your images are stored
train_dir = r"/content/drive/MyDrive/Colab_Notebooks/png/cross_toe_touch-Copy/train_data"
test_dir = r"/content/drive/MyDrive/Colab_Notebooks/png/cross_toe_touch-Copy/test_data"

# Map directory names to integer labels
class_label_mapping = {
    'class_A': 0,
    'class_B': 1,
    'cross_toe_touch': 2,
    'jogging': 3,
    # Add more classes and their corresponding labels if needed
}
# Function to load images and preprocess them
def load_and_preprocess_images(directory):
    images = []
    labels = []
    for label in os.listdir(directory):
        label_dir = os.path.join(directory, label)
        for filename in os.listdir(label_dir):
            img_path = os.path.join(label_dir, filename)
            img = load_img(img_path, target_size=(28, 28))  # Resize images to the desired size
            img_array = img_to_array(img) / 255.0  # Convert image to array and normalize pixel values
            images.append(img_array)
            # Map the directory name to the corresponding integer label
            labels.append(class_label_mapping[label])
    return np.array(images), np.array(labels)

# Load and preprocess the training and test data
x_train, y_train = load_and_preprocess_images(train_dir)
x_test, y_test = load_and_preprocess_images(test_dir)

# Create the CNN model
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 3)),  # Replace '3' with the number of color channels in your images (e.g., 1 for grayscale)
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(2, activation='softmax')  # Replace '2' with the number of classes in your dataset
])

# Compile the model
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Train the model
model.fit(x_train, y_train, epochs=5, batch_size=32, validation_data=(x_test, y_test))

# Evaluate the model on the test set
test_loss, test_accuracy = model.evaluate(x_test, y_test)
print(f"Test accuracy: {test_accuracy}")


Epoch 1/5
1/1 [==============================] - 11s 11s/step - loss: nan - accuracy: 0.0000e+00 - val_loss: nan - val_accuracy: 0.0000e+00
Epoch 2/5
1/1 [==============================] - 0s 37ms/step - loss: nan - accuracy: 0.0000e+00 - val_loss: nan - val_accuracy: 0.0000e+00
Epoch 3/5
1/1 [==============================] - 0s 34ms/step - loss: nan - accuracy: 0.0000e+00 - val_loss: nan - val_accuracy: 0.0000e+00
Epoch 4/5
1/1 [==============================] - 0s 35ms/step - loss: nan - accuracy: 0.0000e+00 - val_loss: nan - val_accuracy: 0.0000e+00
Epoch 5/5
1/1 [==============================] - 0s 27ms/step - loss: nan - accuracy: 0.0000e+00
Test accuracy: 0.0


In [ ]:
from keras.models import load_model
import cv2
import numpy as np
import glob

def writeResultOnImage(openCVImage, resultText):
    # ToDo: this function may take some further fine-tuning to show the text well given any possible image size
    SCALAR_BLUE = (255.0, 0.0, 0.0)
    imageHeight, imageWidth, sceneNumChannels = openCVImage.shape

    # choose a font
    fontFace = cv2.FONT_HERSHEY_TRIPLEX

    # chose the font size and thickness as a fraction of the image size
    fontScale = 4.0
    fontThickness = 3

    # make sure font thickness is an integer, if not, the OpenCV functions that use this may crash
    fontThickness = int(fontThickness)

    upperLeftTextOriginX = int(imageWidth * 0.05)
    upperLeftTextOriginY = int(imageHeight * 0.05)

    textSize, baseline = cv2.getTextSize(resultText, fontFace, fontScale, fontThickness)
    textSizeWidth, textSizeHeight = textSize

    # calculate the lower left origin of the text area based on the text area center, width, and height
    lowerLeftTextOriginX = upperLeftTextOriginX
    lowerLeftTextOriginY = upperLeftTextOriginY + textSizeHeight

    # write the text on the image
    cv2.putText(openCVImage, resultText, (lowerLeftTextOriginX, lowerLeftTextOriginY), fontFace, fontScale, SCALAR_BLUE, fontThickness)
# end function

# dimensions of our images
model=load_model('models/model3classesDogCatBirdAdam32.h5')
model.load_weights('models/checkpoints/weightsAdam32.h5')
#model.summary()

images= glob.glob("data/test/*.jpg")

for i in images:
    image = cv2.imread(i)
    imgr = cv2.resize(image, (150, 150))
    img = np.reshape(imgr, [1, 150, 150, 3])
    classes = model.predict(img)
    classification=np.argmax(classes)
    cv2.namedWindow('Prediction',cv2.WINDOW_NORMAL)
    cv2.resizeWindow('Prediction', 500,500)
    print(classes)
    print(classification)
    prediction=''
    if classification==1:
        prediction='Cat'
    elif classification==0:
        prediction='bird'
    elif classification==2:
        prediction=' Dog'

    writeResultOnImage(image,prediction)
    cv2.imshow('Prediction', image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()


In [8]:
%load_ext tensorboard

In [8]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Conv2D
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Dense


# Define the dimensions of our images.
img_width, img_height = 150, 150

train_data_dir = "/content/drive/MyDrive/Colab_Notebooks/png/cross_toe_touch-Copy/train_data"
validation_data_dir = "/content/drive/MyDrive/Colab_Notebooks/png/cross_toe_touch-Copy/test_data"

# Used to rescale the pixel values from [0, 255] to [0, 1] interval
datagen = ImageDataGenerator(rescale=1./255)

# Automagically retrieve images and their classes for the train and validation sets
train_generator = datagen.flow_from_directory(
        train_data_dir,
        target_size=(img_width, img_height),
        batch_size=16,
        class_mode='categorical',
        shuffle=True)

validation_generator = datagen.flow_from_directory(
        validation_data_dir,
        target_size=(img_width, img_height),
        batch_size=32,
        class_mode='categorical',
        shuffle=True)

model = models.Sequential()
model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(img_width, img_height, 3)))
model.add(MaxPooling2D((2, 2)))

model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D((2, 2)))

model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(MaxPooling2D((2, 2)))

model.add(Flatten())

model.add(Dense(64, activation='relu'))
model.add(layers.Dropout(0.5))

model.add(Dense(3, activation='softmax')) # Replace '3' with the number of classes in your dataset

model.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

nb_epoch = 20
nb_train_samples = 37500
nb_validation_samples = 1245

model.fit(
    train_generator,
    epochs=nb_epoch,
    validation_data=validation_generator,
    validation_steps=nb_validation_samples // validation_generator.batch_size
)

model.save('models/model3classesDogCatBirdAdam32.h5')

print(model.evaluate_generator(validation_generator, nb_validation_samples))


Found 8 images belonging to 2 classes.
Found 4 images belonging to 2 classes.
Epoch 1/20


InvalidArgumentError: ignored